# ChemBreak21 — Cloud Notebook

Run this notebook **from Cell 1 downward**. CB21 uses the uploaded 28-prompt dataset, Gemini 3.1 Pro Preview as the Attack LLM, Gemini 3.8 Flash as the Judge LLM, and separate controller state for ChemDFM and ChemLLM.


In [ ]:
# Cell 1 — user-visible experiment controls
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak21"
EXPERIMENT_REVISION = "CB21_REPLAY_MDP_PROMPTS28_V1"
LIVE                = True
LIVE_PROGRESS       = True
TARGETS             = ["ChemDFM", "ChemLLM"]
print(PROJECT_ID, PROJECT_SUBDIR, EXPERIMENT_REVISION, TARGETS)


In [ ]:
# Cell 2 — clone or refresh repository
import os, subprocess, pathlib
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
REPO_ROOT = pathlib.Path(f"/content/{PROJECT_SUBDIR}_repo")
if REPO_ROOT.exists():
    subprocess.run(["git","-C",str(REPO_ROOT),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"checkout",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin",BRANCH],check=True)
else:
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)],check=True)
PROJECT_ROOT = REPO_ROOT / PROJECT_SUBDIR
assert PROJECT_ROOT.exists(), PROJECT_ROOT
print("Project root:", PROJECT_ROOT)


In [ ]:
# Cell 3 — install pinned dependencies and activate the local src package
import subprocess, sys, importlib
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT_ROOT/"requirements-cloud-ml.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PROJECT_ROOT)],check=True)
SRC_ROOT = PROJECT_ROOT / "src"
PACKAGE_ROOT = SRC_ROOT / "chembreak21"
assert PACKAGE_ROOT.exists(), f"ChemBreak21 source package not found: {PACKAGE_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()
import chembreak21
print("ChemBreak21 import OK:", chembreak21.__version__, chembreak21.__file__)


In [ ]:
# Cell 4 — isolated CB21 storage, caches, GPU
import os, pathlib
STORAGE_ROOT = pathlib.Path("/content/chembreak21_storage")
for d in [STORAGE_ROOT, STORAGE_ROOT/"cache"/"huggingface"/"hub", STORAGE_ROOT/"offload"/"ChemDFM", STORAGE_ROOT/"offload"/"ChemLLM", STORAGE_ROOT/"runs", STORAGE_ROOT/"policies"]:
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(STORAGE_ROOT/"cache"/"huggingface"/"hub")
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
try:
    import torch
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0), "BF16:", torch.cuda.is_bf16_supported())
except Exception as e:
    print("Torch check:", e)


In [ ]:
# Cell 5 — create runtime config without changing the committed config
import yaml
base_config = PROJECT_ROOT / "configs" / "config.cb21.yaml"
cfg = yaml.safe_load(base_config.read_text())
cfg["run"]["dry_run"] = not LIVE
cfg["run"]["live_progress"] = LIVE_PROGRESS
cfg["run"]["experiment_revision"] = EXPERIMENT_REVISION
runtime_config = PROJECT_ROOT / "configs" / "config.cb21.runtime.yaml"
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Runtime config:", runtime_config)
print("Attack LLM:", cfg["roles"]["attack_llm"]["model"])
print("Judge LLM:", cfg["roles"]["judge_llm"]["model"])
print("Targets:", [x["id"] for x in cfg["targets"]])


In [ ]:
# Cell 6 — dataset/config/model preflight
from chembreak21.preflight import run_preflight
report = run_preflight(runtime_config, probe_tokenizers=LIVE, probe_roles=LIVE)
report


In [ ]:
# Cell 7 — verify exact 28-prompt dataset and immutable source-derived anchors
from chembreak21.dataset import selected_tasks
tasks = selected_tasks(PROJECT_ROOT/"data"/"prompts.csv", PROJECT_ROOT/"data"/"CB21_prompts28_manifest_v1.csv")
print("Tasks:", len(tasks))
print("Functional categories:", tasks.functional_category.value_counts().to_dict())
print("Semantic categories:", tasks.semantic_category.value_counts().to_dict())
assert len(tasks)==28 and tasks.original_prompt.notna().all() and tasks.goal_intent_anchor.notna().all()
print("Dataset verification OK")


## Execution semantics

For each target CB21 performs **Baseline → Epoch 1 → Epoch 2 → Epoch 3 → Freeze → Final adaptive attack**. Each epoch starts a fresh target conversation. E2/E3 replay the highest-ranked exact successful attacker path first; if replay fails before the four-turn budget is exhausted, unused turns become adaptive recovery turns. Final evaluation replays complete successful routes in fresh conversations, then uses two frozen-evidence synthesized candidates only if stored routes fail.


In [ ]:
# Cell 8 — helper: run one target completely, then unload it
import json
from chembreak21.runner import ChemBreak21Runner
def run_target(target_id):
    runner = ChemBreak21Runner(runtime_config, target_id)
    try:
        summary = runner.run_all()
        print(json.dumps(summary, indent=2, sort_keys=True))
        return summary
    finally:
        runner.close()


In [ ]:
# Cell 9 — ChemDFM
summary_chemdfm = run_target("ChemDFM")


In [ ]:
# Cell 10 — ChemLLM
summary_chemllm = run_target("ChemLLM")


In [ ]:
# Cell 11 — compact cross-target summary
import pandas as pd
rows=[]
for name,s in [("ChemDFM",summary_chemdfm),("ChemLLM",summary_chemllm)]:
    rows.append({
        "target":name,
        "baseline_asr":s.get("baseline",{}).get("asr"),
        "epoch1_asr":s.get("learning_epoch_1",{}).get("asr"),
        "epoch2_asr":s.get("learning_epoch_2",{}).get("asr"),
        "epoch3_asr":s.get("learning_epoch_3",{}).get("asr"),
        "final_asr":s.get("terminal",{}).get("asr"),
        "actual_target_queries":s.get("actual_target_queries"),
        "replay_turns":s.get("replay_turns"),
        "recovery_turns":s.get("adaptive_recovery_turns"),
    })
pd.DataFrame(rows)


In [ ]:
# Cell 12 — package PUBLIC/SHAREABLE release results only
# Raw state.sqlite3 and internal route-memory JSON remain under chembreak21_storage for private audit.
import shutil, pathlib
public_root = pathlib.Path("/content/cb21_public_results")
if public_root.exists(): shutil.rmtree(public_root)
public_root.mkdir(parents=True)
run_root = STORAGE_ROOT / "runs" / EXPERIMENT_REVISION
for target in TARGETS:
    src = run_root / target / "release"
    if src.exists(): shutil.copytree(src, public_root/target)
zip_path = shutil.make_archive(f"/content/{EXPERIMENT_REVISION}_PUBLIC_results", "zip", root_dir=public_root)
print("Public results ZIP:", zip_path)
print("Internal raw audit remains at:", run_root)


### Interpretation

Because the same 28 source prompts are used across E1–E3 and Final, CB21 measures **within-task adaptive discovery, replication, recovery, and exploitation**, not unseen-prompt generalization. ChemDFM and ChemLLM keep completely separate controller and trajectory memory.
